### Fire Progression Map – Incident Selector

This notebook provides a simple UI to browse wildfire incidents and generate fire progression maps.  
It reads an incident index from `../data/great_basin_ir_2015_2025/ir_events_index.csv` and uses `fire_progression_map.py` to build HTML maps.


#### 1. Setup


In [ ]:
import sys
import subprocess
from pathlib import Path

REQ_FILE = Path("../requirements.txt")

if REQ_FILE.exists():
    print(f"[setup] Installing/updating packages from {REQ_FILE} …")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)]
    )
    print("[setup] Done.")
else:
    print(f"[setup] WARNING: {REQ_FILE} not found. Skipping package installation.")


[setup] Installing / updating packages from ..\requirements.txt …
[setup] Done.


In [4]:
# packages 

from pathlib import Path
import sys

import pandas as pd
from ipywidgets import ToggleButtons, VBox, HBox, Output, Button
from IPython.display import display


In [2]:
# Year + incident toggle UI for fire progression map

# make sure we can import fire_progression_map.py
module_dir = Path(r"..\modules")
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from fire_progression_map import build_fire_progression_map

# paths to data 
data_root = Path(r"..\data\great_basin_ir_2015_2025")
index_csv = Path(r"..\data\great_basin_ir_2015_2025\ir_events_index.csv")

print(f"data_root: {data_root}")
print(f"index_csv: {index_csv}")
assert index_csv.exists(), f"Index CSV not found at {index_csv}"

# optional: where to save the HTML maps
# out_dir = Path("maps")
# out_dir.mkdir(exist_ok=True, parents=True)

# load index once 
df_index = pd.read_csv(index_csv)

# sanity check expected columns
if "incident_year" not in df_index.columns or "incident" not in df_index.columns:
    raise KeyError("Expected columns 'incident_year' and 'incident' in index CSV")

df_index["incident_year"] = df_index["incident_year"].astype(int)

# year toggle
year_options = sorted(df_index["incident_year"].unique())
year_toggle = ToggleButtons(
    options=year_options,
    description="Year:",
    button_style="",  # '', 'primary', 'success', 'info', 'warning', 'danger'
)

#incident toggle (populated dynamically from year)
incident_toggle = ToggleButtons(
    options=[],
    description="Incident:",
    button_style="",
    layout={"width": "600px"},
)

#run button + output area
run_btn = Button(
    description="Generate map",
    button_style="primary",
    tooltip="Build fire progression map for the selected incident",
)
map_out = Output()


NameError: name 'Path' is not defined

#### 2. Select incident and generate map

The controls below let you:

- Choose a **year** using the toggle buttons.
- Pick an **incident** from that year.
- Click **“Generate map”** to call `build_fire_progression_map(...)`.


In [ ]:
def update_incident_options(*args):
    """Update incident toggle when year changes."""
    year = year_toggle.value
    df_y = df_index[df_index["incident_year"] == year]

    incident_options = sorted(df_y["incident"].unique())
    incident_toggle.options = incident_options

    if incident_options:
        incident_toggle.value = incident_options[0]


def on_run_clicked(_):
    """Build and display the map for the current selection."""
    map_out.clear_output()

    year = int(year_toggle.value)
    incident = incident_toggle.value

    # same naming logic as before
    safe_incident = "".join(
        c for c in incident if c.isalnum() or c in (" ", "_", "-")
    ).strip().replace(" ", "_")
    out_html = out_dir / f"{year}_{safe_incident}_progression.html"

    with map_out:
        print(f"Building fire map for {year} — {incident}…")
        m = build_fire_progression_map(
            data_root=data_root,
            index_csv=index_csv,
            year=year,
            incident_query=incident,
            out_html=out_html,
        )
        display(m)
        print(f"Saved HTML to: {out_html.resolve()}")


# link callbacks
year_toggle.observe(update_incident_options, names="value")
run_btn.on_click(on_run_clicked)

# initialize incident choices with first year
update_incident_options()

# show the controls + map
display(
    VBox(
        [
            HBox([year_toggle]),
            HBox([incident_toggle]),
            run_btn,
            map_out,
        ]
    )
)